# 第309章: エージング・シミュレーション

## 📋 この章で学ぶこと

この章を終えると、以下ができるようになります：

- [ ] 潜在空間の属性編集技術を応用して「老化シミュレーション」を実装できる
- [ ] 数字画像の連続的な劣化（摩耗、ぼやけ、ノイズ）を潜在空間操作で実現できる
- [ ] 複数の属性を同時に時間的に変化させるアニメーションを作成できる
- [ ] 画像変容の品質を定量的に評価できる（FID概念、SSIM、PSNR）

## 🎯 前提知識

- ✅ Notebook 302（ベクトル演算）、307（InterFaceGAN）、308（GANSpace）
- ✅ Notebook 306（Slerp補間）
- ✅ Phase 2の全体的な理解

⏱️ **推定学習時間**: 90-120分
📊 **難易度**: ★★★★☆（上級）
🎓 **カテゴリ**: 実践・応用

---

## 🌟 はじめに — Phase 2の集大成

この章では、Phase 2で学んだ技術を統合して**実践的な画像変容タスク**に取り組みます。

### テーマ: 数字の「エージング」（老化・劣化）

手書き数字が時間の経過とともに「劣化」していくプロセスをシミュレートします：

```
  時間 t=0          t=0.3          t=0.6          t=1.0
  ┌─────┐         ┌─────┐         ┌─────┐         ┌─────┐
  │     │         │     │         │     │         │     │
  │  3  │   →→    │  3  │   →→    │  3  │   →→    │  3  │
  │     │         │  ·  │         │ ·:· │         │ :·: │
  │きれい│         │少し劣化│       │かなり劣化│     │ぼやけ│
  └─────┘         └─────┘         └─────┘         └─────┘
```

### 使う技術

| 技術 | 役割 | 出典 |
|------|------|------|
| 属性ベクトル | 「劣化方向」の発見 | 307章 InterFaceGAN |
| PCA主成分 | 追加の変動方向 | 308章 GANSpace |
| Slerp補間 | 滑らかな変化 | 306章 |
| 複数属性同時操作 | 複合的な劣化表現 | 302章 |

### 📝 この章の構成

1. **劣化属性の定義** — 太さ減少 + ノイズ + ぼやけ
2. **単一属性のエージング** — 1つの劣化要因を時間変化
3. **複合エージング** — 複数の劣化を同時進行
4. **アニメーション生成** — エージング過程のGIF
5. **品質評価** — SSIM/PSNRで変化を定量化

In [ ]:
# ============================================================
# 環境設定
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.animation as animation
from IPython.display import HTML, Image as IPImage
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

def setup_japanese_font():
    japanese_fonts = ['Hiragino Sans', 'Yu Gothic', 'MS Gothic', 'Noto Sans CJK JP', 'IPAexGothic']
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 8)

print(f"Device: {device}")
print("✅ 環境設定完了")

In [ ]:
# ============================================================
# VAE学習 + 属性方向の準備
# ============================================================

class VAE(nn.Module):
    def __init__(self, latent_dim=20):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU())
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
        )
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), self.fc_logvar(h)
    def decode(self, z): return self.decoder(z)
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(logvar)
        return self.decode(z), mu, logvar

transform = transforms.ToTensor()
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

model = VAE(latent_dim=20).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print("VAE (z=20) 学習中...")
for epoch in range(20):
    model.train()
    total = 0
    for x, _ in train_loader:
        x = x.view(-1, 784).to(device)
        recon, mu, logvar = model(x)
        loss = nn.functional.binary_cross_entropy(recon, x, reduction='sum') \
               - 0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1}/20 | Loss: {total/len(train_ds):.2f}")

model.eval()
all_z, all_labels, all_images = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        mu, _ = model.encode(x.view(-1, 784).to(device))
        all_z.append(mu.cpu().numpy())
        all_labels.append(y.numpy())
        all_images.append(x.numpy())
all_z = np.concatenate(all_z)
all_labels = np.concatenate(all_labels)
all_images = np.concatenate(all_images)
digit_indices = {d: np.where(all_labels == d)[0] for d in range(10)}

print("✅ 学習・エンコード完了")

---

## 1. 劣化属性の定義

「エージング」を構成する劣化要因を潜在空間内の方向として定義します。

| 劣化要因 | 物理的意味 | 潜在空間での表現 |
|---------|----------|----------------|
| **線の細化** | インクの薄れ、摩耗 | 太さベクトルの逆方向 |
| **ぼやけ** | 紙の劣化、にじみ | ぼやけ方向ベクトル |
| **ノイズ** | 汚れ、傷 | 潜在空間にランダム摂動 |
| **変形** | 紙の反り | GANSpace PC方向 |

In [ ]:
# ============================================================
# 劣化属性ベクトルの計算
# ============================================================

def decode_z(z_vec):
    with torch.no_grad():
        z_t = torch.tensor(z_vec, dtype=torch.float32).unsqueeze(0).to(device)
        return model.decode(z_t).cpu().numpy().reshape(28, 28)

# 1. 太さ方向（InterFaceGAN方式）
imgs_flat = all_images.reshape(len(all_images), -1)
ink_amounts = imgs_flat.sum(axis=1)
thickness_label = (ink_amounts > np.median(ink_amounts)).astype(int)

scaler = StandardScaler()
z_scaled = scaler.fit_transform(all_z)
svm_thick = LinearSVC(C=1.0, max_iter=5000, random_state=42)
svm_thick.fit(z_scaled, thickness_label)
d_thickness = svm_thick.coef_[0] / scaler.scale_
d_thickness = d_thickness / np.linalg.norm(d_thickness)

# 2. ぼやけ方向（高周波成分の量で分類）
blur_scores = []
for img in all_images.reshape(-1, 28, 28):
    # ラプラシアンの分散 = シャープネス指標
    laplacian = np.abs(np.diff(img, axis=0)[:, :-1]) + np.abs(np.diff(img, axis=1)[:-1, :])
    blur_scores.append(laplacian.mean())
blur_scores = np.array(blur_scores)
blur_label = (blur_scores > np.median(blur_scores)).astype(int)

svm_blur = LinearSVC(C=1.0, max_iter=5000, random_state=42)
svm_blur.fit(z_scaled, blur_label)
d_blur = svm_blur.coef_[0] / scaler.scale_
d_blur = d_blur / np.linalg.norm(d_blur)

# 3. GANSpace方向（PCA主成分）
from sklearn.decomposition import PCA
pca = PCA(n_components=20)
pca.fit(all_z)
d_deform = pca.components_[2]  # 第3主成分を変形方向として使用

print("✅ 劣化属性ベクトル計算完了")
print(f"  太さ SVM精度: {svm_thick.score(z_scaled, thickness_label):.3f}")
print(f"  ブラー SVM精度: {svm_blur.score(z_scaled, blur_label):.3f}")

In [ ]:
# ============================================================
# 単一属性のエージング（時間変化）
# ============================================================

n_time = 12
time_steps = np.linspace(0, 1, n_time)
digits_demo = [0, 3, 5, 7, 9]

fig, axes = plt.subplots(len(digits_demo), n_time, figsize=(18, len(digits_demo) * 1.8))

for row, digit in enumerate(digits_demo):
    idx = digit_indices[digit][0]
    z_base = all_z[idx]

    for col, t in enumerate(time_steps):
        # 太さを時間とともに減少させる（エージング = 線が消えていく）
        z_aged = z_base - t * 2.5 * d_thickness
        img = decode_z(z_aged)
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(f'{digit}', fontsize=14, rotation=0, labelpad=20)
        if row == 0:
            axes[row, col].set_title(f't={t:.1f}', fontsize=9)

fig.suptitle('単一属性エージング: 線の細化\nt=0（元画像）→ t=1.0（最大劣化）',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_309_01_single_aging.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 時間 t が増えるにつれて線が全体的に細くなっていく")
print("   これは「インクが薄れる」老化プロセスの模倣")

---

## 2. 複合エージング — 複数の劣化を同時進行

実際の「老化」は複数の要因が同時に進行します。
各要因を異なる速度で変化させることで、よりリアルなエージングを表現します。

```
  時間 t:  0 ──────────────────→ 1.0
  
  線の細化:  ─── ゆっくり開始  ────→  加速
  ぼやけ:    ─── 早期に開始   ────→  飽和
  変形:      ───── 後半で開始 ────→  増加
  ノイズ:    ─────── 後半で ───→  急増
```

In [ ]:
# ============================================================
# 複合エージング関数
# ============================================================

def aging_schedule(t, mode='linear'):
    """劣化の進行スケジュール"""
    schedules = {
        'thin': 1 - np.cos(t * np.pi / 2),     # 最初はゆっくり、後半加速
        'blur': np.sqrt(t),                      # 早期に効果が出る
        'deform': t ** 2,                        # 後半で急に効果
        'noise': t ** 3,                         # 最後にノイズが増える
    }
    return schedules

def apply_aging(z_base, t, strength=2.0):
    """複合エージングを適用"""
    sched = aging_schedule(t)

    z_aged = z_base.copy()
    z_aged -= sched['thin'] * strength * d_thickness    # 線が細くなる
    z_aged -= sched['blur'] * strength * 0.5 * d_blur   # ぼやける
    z_aged += sched['deform'] * strength * 0.3 * d_deform  # 変形
    z_aged += sched['noise'] * np.random.randn(len(z_base)) * 0.15  # ノイズ

    return z_aged

# スケジュールの可視化
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
t_axis = np.linspace(0, 1, 100)
for t_val in t_axis:
    sched = aging_schedule(t_val)

colors = {'thin': '#e74c3c', 'blur': '#3498db', 'deform': '#2ecc71', 'noise': '#9b59b6'}
labels = {'thin': '線の細化', 'blur': 'ぼやけ', 'deform': '変形', 'noise': 'ノイズ'}

for key in ['thin', 'blur', 'deform', 'noise']:
    values = [aging_schedule(t)[key] for t in t_axis]
    ax.plot(t_axis, values, color=colors[key], linewidth=2.5, label=labels[key])

ax.set_xlabel('時間 t', fontsize=14)
ax.set_ylabel('劣化強度', fontsize=14)
ax.set_title('エージングスケジュール\n各劣化要因の進行曲線', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_309_02_aging_schedule.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 複合エージングの実行
# ============================================================

n_time = 12
time_steps = np.linspace(0, 1, n_time)
digits_demo = [0, 3, 5, 7, 9]

fig, axes = plt.subplots(len(digits_demo), n_time, figsize=(18, len(digits_demo) * 1.8))

np.random.seed(42)
for row, digit in enumerate(digits_demo):
    idx = digit_indices[digit][0]
    z_base = all_z[idx]

    for col, t in enumerate(time_steps):
        z_aged = apply_aging(z_base, t, strength=2.0)
        img = decode_z(z_aged)
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(f'{digit}', fontsize=14, rotation=0, labelpad=20)
        if row == 0:
            axes[row, col].set_title(f't={t:.2f}', fontsize=9)

fig.suptitle('複合エージング: 細化 + ぼやけ + 変形 + ノイズ\nt=0（新品）→ t=1.0（劣化）',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_309_03_composite_aging.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 複数の劣化要因が時間差で進行するため、より自然な「老化」に見える")

In [ ]:
# ============================================================
# エージング・アニメーションの生成
# ============================================================

np.random.seed(42)

# 静止画版（GIF代替）
fig, axes = plt.subplots(3, 16, figsize=(20, 4.5))
n_frames = 16
t_anim = np.linspace(0, 1, n_frames)

for row, digit in enumerate([3, 5, 8]):
    idx = digit_indices[digit][0]
    z_base = all_z[idx]

    for col, t in enumerate(t_anim):
        z_aged = apply_aging(z_base, t, strength=2.5)
        img = decode_z(z_aged)
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'{digit}', fontsize=14, rotation=0, labelpad=15)

fig.suptitle('エージング・アニメーション（静止画版）\n← 新品   劣化 →',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_309_04_aging_animation.png', dpi=150, bbox_inches='tight')
plt.show()

# GIFアニメーション
try:
    digit = 3
    idx = digit_indices[digit][0]
    z_base = all_z[idx]

    np.random.seed(42)
    fig_gif, ax_gif = plt.subplots(1, 1, figsize=(3, 3))
    frames = []
    for t in np.linspace(0, 1, 30):
        z_aged = apply_aging(z_base, t, strength=2.5)
        img = decode_z(z_aged)
        im = ax_gif.imshow(img, cmap='gray', animated=True)
        title = ax_gif.text(0.5, 1.05, f't={t:.2f}', ha='center',
                           transform=ax_gif.transAxes, fontsize=12)
        ax_gif.axis('off')
        frames.append([im, title])

    anim = animation.ArtistAnimation(fig_gif, frames, interval=100, blit=True)
    anim.save('fig_309_05_aging.gif', writer='pillow', fps=10)
    plt.close(fig_gif)
    print("✅ GIF保存: fig_309_05_aging.gif")
    display(IPImage(filename='fig_309_05_aging.gif'))
except Exception as e:
    plt.close(fig_gif)
    print(f"⚠️ GIF保存スキップ: {e}")

---

## 3. 品質評価 — 変化の定量化

エージングの効果を数値で測定するため、画像品質指標を使います。

| 指標 | 何を測るか | 高い/低いの意味 |
|------|----------|--------------|
| **SSIM** | 構造的類似度 | 高い＝元画像に似ている |
| **PSNR** | 信号対雑音比 | 高い＝ノイズが少ない |
| **ピクセル密度** | インク量 | 低い＝線が消えている |

In [ ]:
# ============================================================
# エージング過程の品質指標
# ============================================================

def ssim_simple(img1, img2):
    """簡易SSIM計算"""
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    mu1, mu2 = img1.mean(), img2.mean()
    var1, var2 = img1.var(), img2.var()
    cov = ((img1 - mu1) * (img2 - mu2)).mean()
    num = (2 * mu1 * mu2 + c1) * (2 * cov + c2)
    den = (mu1**2 + mu2**2 + c1) * (var1 + var2 + c2)
    return num / den

def psnr(img1, img2):
    """PSNR計算"""
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0: return float('inf')
    return 10 * np.log10(1.0 / mse)

# エージング過程の指標を計算
n_time_eval = 20
t_eval = np.linspace(0, 1, n_time_eval)
digits_eval = [0, 3, 5, 7, 9]

np.random.seed(42)
metrics = {d: {'ssim': [], 'psnr': [], 'density': []} for d in digits_eval}

for digit in digits_eval:
    idx = digit_indices[digit][0]
    z_base = all_z[idx]
    img_orig = decode_z(z_base)

    for t in t_eval:
        z_aged = apply_aging(z_base, t, strength=2.0)
        img_aged = decode_z(z_aged)

        metrics[digit]['ssim'].append(ssim_simple(img_orig, img_aged))
        metrics[digit]['psnr'].append(psnr(img_orig, img_aged))
        metrics[digit]['density'].append(img_aged.sum())

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_d = plt.cm.tab10(np.linspace(0, 1, len(digits_eval)))

for i, digit in enumerate(digits_eval):
    axes[0].plot(t_eval, metrics[digit]['ssim'], color=colors_d[i], linewidth=2, label=f'{digit}')
    axes[1].plot(t_eval, metrics[digit]['psnr'], color=colors_d[i], linewidth=2, label=f'{digit}')
    axes[2].plot(t_eval, metrics[digit]['density'], color=colors_d[i], linewidth=2, label=f'{digit}')

titles = ['SSIM（構造的類似度）', 'PSNR（信号対雑音比）', 'ピクセル密度（インク量）']
ylabels = ['SSIM', 'PSNR (dB)', '密度']

for ax, title, ylabel in zip(axes, titles, ylabels):
    ax.set_xlabel('時間 t', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

fig.suptitle('エージング過程の品質指標', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_309_06_quality_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 SSIMとPSNRは時間とともに低下 → 元画像からの乖離が増加")
print("   ピクセル密度の低下 → インクが消えていく効果を確認")

---

## まとめ

### 🎯 このノートブックで学んだこと

**エージング・シミュレーション**
- ✓ 複数の劣化属性（細化、ぼやけ、変形、ノイズ）を組み合わせた複合劣化
- ✓ 時間変化のスケジュール設計（各要因の進行曲線）

**品質評価**
- ✓ SSIM、PSNRで画像変容の度合いを定量化
- ✓ ピクセル密度で劣化の物理的効果を測定

**Phase 2の統合**
- ✓ InterFaceGAN（属性方向）、GANSpace（PCA方向）、ベクトル演算を統合的に活用
- ✓ 潜在空間操作による画像変容の実践力を獲得

### ⚠️ よくある間違い

❌ 「エージングは単にノイズを加えるだけ」
✅ 効果的なエージングは、複数の属性を異なる速度で変化させ、時間スケジュールを設計する必要がある

❌ 「潜在空間操作だけでリアルな劣化を表現できる」
✅ VAEの潜在空間はMNISTの学習データの範囲でしか表現できない。データにない劣化パターンは生成不可能

### ✅ 学習チェックリスト

- [ ] 複数の劣化要因を潜在空間で同時操作できるか？
- [ ] エージングスケジュールの設計意図を説明できるか？
- [ ] SSIMとPSNRの違いを説明できるか？

---

**次のステップ**: ノートブック310「DDIM反転入門」で、Phase 3（拡散モデルベースの画像変容）に進みます！ 拡散モデルの逆過程を使った全く新しいアプローチを学びます。

---

## 🎓 自己評価クイズ

### Q1: エージングスケジュールで「線の細化」を後半加速にし、「ぼやけ」を早期開始にした理由は？

<details>
<summary>💡 答えを見る</summary>

**答え**: 物理的な劣化プロセスを模倣するため

実際のインクの劣化では、最初ににじみ（ぼやけ）が起き、その後時間が経つにつれて線が消えていく。
スケジュールは cos(πt/2) や √t などの関数で、自然な進行速度を表現している。

</details>

---

### Q2: SSIMとPSNRの違いは何か？どちらがより人間の知覚に近いか？

<details>
<summary>💡 答えを見る</summary>

**答え**: SSIMのほうが人間の知覚に近い

PSNRはピクセル単位のMSEに基づく単純な指標。SSIMは輝度、コントラスト、構造の3つを考慮し、人間の視覚システムに近い。例えば、画像全体が少しシフトするとPSNRは大きく下がるが、SSIMはあまり下がらない。

</details>

---

### Q3: 潜在空間にランダム摂動（ノイズ）を加えるのと、画像にノイズを加えるのは何が違うか？

<details>
<summary>💡 答えを見る</summary>

**答え**: 潜在空間のノイズは「意味的な」ランダム変動を生み、画像のノイズは「ピクセルレベルの」ランダム変動を生む

潜在空間のノイズは、形状の微小変化、太さの揺らぎなど、学習データに含まれる自然なバリエーションを引き起こす。一方、画像に直接ノイズを加えると、データ分布に存在しない不自然なパターンが生じる。

</details>